System construction and test


In [2]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
import numpy as np
#from Alert import Alert
pd.options.mode.chained_assignment = None  # default='warn'
#alert = Alert('1h','GGAL')

input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [72]:
dataini = '2015-04-03 19:30:00'
datafim = '2019-04-03 19:30:00'

In [74]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo","volume"])
display(len(dftitulosdados))
#display(dftitulosdados.head(10))

8180

In [9]:

i = 'high'
K = 16  
D = 5    
smoth = 5  
medM = 8  
lowM = 10
stpl = 0.02
comission = 0.0035
taxalivrerisgoprom = 0.05
drawmax = -0.1


In [21]:
%%timeit
# Stochastic calculation
def stochastic(dftitulosdados, i, K, D, smoth):
    df = dftitulosdados    
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i ] = df.k.rolling(smoth).mean()
    df["d" + i ] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  
    dfstoch = df
    return dfstoch



226 ns ± 24.6 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [13]:
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)
display (dfstoch.head(10))

,datetime,open,high,low,close,khigh,dhigh
0,2015-04-06 09:00:00,18.5374,18.7714,18.3541,18.3659,NaN,NaN
1,2015-04-06 10:00:00,18.3737,18.5452,18.2411,18.5062,NaN,NaN
2,2015-04-06 11:00:00,18.5218,18.5998,18.4828,18.5452,NaN,NaN
3,2015-04-06 12:00:00,18.5452,18.5920,18.4750,18.5686,NaN,NaN
4,2015-04-06 13:00:00,18.5608,18.6388,18.5452,18.5686,NaN,NaN
5,2015-04-06 14:00:00,18.5686,18.7168,18.5530,18.5530,NaN,NaN
6,2015-04-06 15:00:00,18.5530,18.5764,18.3815,18.4595,NaN,NaN
7,2015-04-06 16:00:00,18.4595,18.4595,18.4595,18.4595,NaN,NaN
8,2015-04-07 09:00:00,18.4283,18.6076,18.4283,18.5998,NaN,NaN
9,2015-04-07 10:00:00,18.5998,18.5998,18.4984,18.5842,NaN,NaN


In [23]:
%%timeit
# stochastic high, med and low frcuency

def stoch_hml( dfstoch, k, d, smth, medM, lowM): 
    df = dfstoch
    df = stochastic(df, "high", k, d, smth)
    df = stochastic(df, "med", k*medM, d*medM, smth*medM)
    df = stochastic(df, "low", k*medM*lowM, d*medM*lowM, smth*medM*lowM)
    dfstoch_hml = df
    return dfstoch_hml
    



244 ns ± 43.3 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [17]:
dfstoch_hml= stoch_hml(dfstoch , K, D, smoth, medM , lowM )
display(dfstoch_hml.head(10) )

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow
0,2015-04-06 09:00:00,18.5374,18.7714,18.3541,18.3659,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-04-06 10:00:00,18.3737,18.5452,18.2411,18.5062,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-04-06 11:00:00,18.5218,18.5998,18.4828,18.5452,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-04-06 12:00:00,18.5452,18.5920,18.4750,18.5686,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-04-06 13:00:00,18.5608,18.6388,18.5452,18.5686,NaN,NaN,NaN,NaN,NaN,NaN
5,2015-04-06 14:00:00,18.5686,18.7168,18.5530,18.5530,NaN,NaN,NaN,NaN,NaN,NaN
6,2015-04-06 15:00:00,18.5530,18.5764,18.3815,18.4595,NaN,NaN,NaN,NaN,NaN,NaN
7,2015-04-06 16:00:00,18.4595,18.4595,18.4595,18.4595,NaN,NaN,NaN,NaN,NaN,NaN
8,2015-04-07 09:00:00,18.4283,18.6076,18.4283,18.5998,NaN,NaN,NaN,NaN,NaN,NaN
9,2015-04-07 10:00:00,18.5998,18.5998,18.4984,18.5842,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
%%timeit
def system_criterias (dfstoch_hml):   # input  df() =  dfstoch_hml ()
    df = dfstoch_hml
    df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
    df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
    df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)
    dfcriterias = df
    return dfcriterias


203 ns ± 17.7 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [27]:
dfcriterias = system_criterias (dfstoch_hml)
display (dfcriterias.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh
0,2015-04-06 09:00:00,18.5374,18.7714,18.3541,18.3659,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2015-04-06 10:00:00,18.3737,18.5452,18.2411,18.5062,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2015-04-06 11:00:00,18.5218,18.5998,18.4828,18.5452,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2015-04-06 12:00:00,18.5452,18.5920,18.4750,18.5686,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2015-04-06 13:00:00,18.5608,18.6388,18.5452,18.5686,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
5,2015-04-06 14:00:00,18.5686,18.7168,18.5530,18.5530,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
6,2015-04-06 15:00:00,18.5530,18.5764,18.3815,18.4595,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
7,2015-04-06 16:00:00,18.4595,18.4595,18.4595,18.4595,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
8,2015-04-07 09:00:00,18.4283,18.6076,18.4283,18.5998,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
9,2015-04-07 10:00:00,18.5998,18.5998,18.4984,18.5842,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


dfsignals = system_signals(dfcriterias)
display (dfsignals.head(10))

In [36]:
%%timeit
def system_signals (dfcriterias):
    df = dfcriterias
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)  # inicializa com "standby"
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "enter"
            estado_anterior = "enter"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif low[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "out":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    dfsignals = df
    return df

212 ns ± 13.5 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [34]:
dfsignals = system_signals (dfcriterias)
display (dfsignals.head(10))

,datetime,open,high,low,close,khigh,dhigh,kmed,dmed,klow,dlow,longbuylow,longbuymed,longbuyhigh,state
0,2015-04-06 09:00:00,18.5374,18.7714,18.3541,18.3659,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
1,2015-04-06 10:00:00,18.3737,18.5452,18.2411,18.5062,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
2,2015-04-06 11:00:00,18.5218,18.5998,18.4828,18.5452,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
3,2015-04-06 12:00:00,18.5452,18.5920,18.4750,18.5686,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
4,2015-04-06 13:00:00,18.5608,18.6388,18.5452,18.5686,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
5,2015-04-06 14:00:00,18.5686,18.7168,18.5530,18.5530,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
6,2015-04-06 15:00:00,18.5530,18.5764,18.3815,18.4595,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
7,2015-04-06 16:00:00,18.4595,18.4595,18.4595,18.4595,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
8,2015-04-07 09:00:00,18.4283,18.6076,18.4283,18.5998,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby
9,2015-04-07 10:00:00,18.5998,18.5998,18.4984,18.5842,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,standby


dfstate = long_buy_crits_numpy(dfstoch_hml)
pd.set_option('display.max_rows', None)
dfbuysellop = dfstate[(dfstate['state'] == 'buylong') | (dfstate['state'] == 'selllong')]
#df_intervalo = dfstate.iloc[800:1500] 
#display(df_intervalo)
display (len(dfbuysell))
display(dfbuysell.head(10))

Stop Loss Reentry

In [42]:
%%timeit
def stop_loss_reentry (dfsignals, stpl) :

    df = dfsignals[(dfsignals['state'] == 'enter') | (dfsignals['state'] == 'stay')]
    df = df.reset_index(drop=True)
    df = df.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    #drop(columns=["symbol", "moeda", "intervalo","volume"])
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "enter" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "stay" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "out"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "enter"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "outstpl"

    return df


197 ns ± 4.74 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [40]:
dfstoploss = stop_loss_reentry (dfsignals, stpl)
display(dfstoploss.head(10))

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,stpl
0,2016-04-13 11:00:00,22.5714,1,1,1,enter,0.451428
1,2016-04-13 12:00:00,22.3762,1,1,1,stay,0.256228
2,2016-04-13 13:00:00,22.6260,1,1,1,stay,0.506028
3,2016-04-13 14:00:00,22.5870,1,1,0,stay,0.467028
4,2016-04-13 15:00:00,22.8134,1,1,0,stay,0.693428
5,2016-04-13 16:00:00,22.8369,1,1,0,stay,0.716928
6,2016-04-14 09:00:00,22.8603,1,1,1,stay,0.740328
7,2016-04-14 10:00:00,22.9462,1,1,1,stay,0.826228
8,2016-04-14 11:00:00,22.8447,1,1,1,stay,0.724728
9,2016-04-14 12:00:00,22.8447,1,1,0,stay,0.724728


In [52]:
#%%timeit
# preparar dataframe para calcular index 
def index_dataframe (dfsignals, dfstoploss) :
    # elimino colunas de dfsignals e filtro por os valores "out"
    dfsignalsdrop = dfsignals.drop(columns=["open" ,"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    dfsignalsout = dfsignalsdrop[(dfsignalsdrop['state'] == 'out')]

    # elimino a culuna stpl de dfstoploss e filtro os valores enter e out 
    dfstoplossdrop = dfstoploss.drop(columns=["stpl"]) 
    dfstoplossenterout = dfstoplossdrop[(dfstoplossdrop['state'] == 'enter') | (dfstoplossdrop['state'] == 'out')]

    # concatenar os dois df para ter o total dos signals enter e out
    dfsignalsenterout = pd.concat([dfsignalsout, dfstoplossenterout], ignore_index=True)

    # Ordenar pelo datetime e resetear o index
    dfsignalsenterout["datetime"] = pd.to_datetime(dfsignalsenterout["datetime"])
    dfsignalsenterout = dfsignalsenterout.sort_values("datetime").reset_index(drop=True)

    # limpar os out duplicados"out" por a saida anticipada do stoploss e reiniciar indice
    df = dfsignalsenterout
    cond = (df["state"] == "out")  & (df["state"].shift(1) == "out")
    dfsignalsentoutclean = df[~cond].reset_index(drop=True)
    return dfsignalsentoutclean


229 ns ± 5.9 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [50]:
dfsignalsentoutclean = index_dataframe (dfsignals, dfstoploss)
display (dfsignalsentoutclean)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state
0,2016-04-13 11:00:00,22.5714,1,1,1,enter
1,2016-04-25 11:00:00,22.9852,1,0,0,out
2,2016-07-14 16:00:00,25.1422,1,1,1,enter
3,2016-07-20 13:00:00,24.7357,1,0,0,out
4,2016-08-09 14:00:00,23.2034,1,1,1,enter
5,2016-08-11 11:00:00,23.0783,0,1,0,out
6,2016-11-07 15:00:00,24.0477,1,1,1,enter
7,2016-11-09 15:00:00,23.3832,1,1,0,out
8,2017-02-07 12:00:00,25.6660,1,1,1,enter
9,2017-02-16 15:00:00,26.7683,1,0,0,out


Index ,Trade, Index sin comission

In [58]:
%%timeit
def index_trade(dfsignalsenteroutclean):
    df = dfsignalsentoutclean
    df ["index_sc"] = 100. 
    df ["trade"] = 0.
    df ["index"] = 100. *(1-comission) 
    for i in range(1, len(df)):      
                       
        if  df.loc[i, "state"] == "out" :
            df.loc[i, "index_sc"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1)* df.loc[i-1,"index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
            df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
            
        if  df.loc[i, "state"] == "enter" :        
            df.loc[i, "index_sc"] =  df.loc[i-1, "index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
    dfindex = df
    return dfindex


188 ns ± 7.29 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [56]:
dfindex = index_trade(dfsignalsentoutclean)
display (dfindex)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,index_sc,trade,index
0,2016-04-13 11:00:00,22.5714,1,1,1,enter,100.000000,0.000000,99.650000
1,2016-04-25 11:00:00,22.9852,1,0,0,out,101.833293,0.018333,101.476877
2,2016-07-14 16:00:00,25.1422,1,1,1,enter,101.833293,0.000000,101.476877
3,2016-07-20 13:00:00,24.7357,1,0,0,out,100.186849,-0.016168,99.836195
4,2016-08-09 14:00:00,23.2034,1,1,1,enter,100.186849,0.000000,99.836195
5,2016-08-11 11:00:00,23.0783,0,1,0,out,99.646697,-0.005391,99.297933
6,2016-11-07 15:00:00,24.0477,1,1,1,enter,99.646697,0.000000,99.297933
7,2016-11-09 15:00:00,23.3832,1,1,0,out,96.893201,-0.027633,96.554075
8,2017-02-07 12:00:00,25.6660,1,1,1,enter,96.893201,0.000000,96.554075
9,2017-02-16 15:00:00,26.7683,1,0,0,out,101.054558,0.042948,100.700867


stop drawdawn

In [64]:
#%%timeit
def stop_drawdawn (dfindex, drawmax):
    df = dfindex
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["index"] = pd.to_numeric(df["index"], errors="coerce")

    estado_corrigido = []
    pico_atual = df.loc[0, "index"]

    for i in range(len(df)):
        valor_index = df.loc[i, "index"]
        estado = df.loc[i, "state"]

        # Atualiza pico se houve recuperação
        if valor_index > pico_atual:
            pico_atual = valor_index

        # Calcula drawdown
        if pico_atual > 0:
            drawdown = (valor_index - pico_atual) / pico_atual
        else:
            drawdown = 0

        # Verifica se deve aplicar stopsys
        if estado == "out" and drawdown < drawmax:
            estado = "stopsys"
            pico_atual = valor_index  # reinicia ciclo a partir desse ponto

        estado_corrigido.append(estado)
    df["state"] = estado_corrigido
    dfindexdrawdawn = df
    return dfindexdrawdawn

194 ns ± 7.24 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [62]:
dfindexdrawdawn = stop_drawdawn(dfindex, drawmax)
display(dfindexdrawdawn)

,datetime,close,longbuylow,longbuymed,longbuyhigh,state,index_sc,trade,index
0,2016-04-13 11:00:00,22.5714,1,1,1,enter,100.000000,0.000000,99.650000
1,2016-04-25 11:00:00,22.9852,1,0,0,out,101.833293,0.018333,101.476877
2,2016-07-14 16:00:00,25.1422,1,1,1,enter,101.833293,0.000000,101.476877
3,2016-07-20 13:00:00,24.7357,1,0,0,out,100.186849,-0.016168,99.836195
4,2016-08-09 14:00:00,23.2034,1,1,1,enter,100.186849,0.000000,99.836195
5,2016-08-11 11:00:00,23.0783,0,1,0,out,99.646697,-0.005391,99.297933
6,2016-11-07 15:00:00,24.0477,1,1,1,enter,99.646697,0.000000,99.297933
7,2016-11-09 15:00:00,23.3832,1,1,0,out,96.893201,-0.027633,96.554075
8,2017-02-07 12:00:00,25.6660,1,1,1,enter,96.893201,0.000000,96.554075
9,2017-02-16 15:00:00,26.7683,1,0,0,out,101.054558,0.042948,100.700867


Metricas

In [169]:
def estatisticas_index(df):
    serie = df["index"].dropna()

    maximo = serie.max()
    minimo = serie.min()
    media = serie.mean()
    desvio = serie.std()
    taxagantot = (maximo - minimo)/minimo

    df["datetime"] = pd.to_datetime(df["datetime"])
    data_inicial = df["datetime"].min()
    data_final = df["datetime"].max()
    dias = (data_final - data_inicial).days
    

    taxagananualprom = (1 + taxagantot) ** (1 / dias * 365) - 1  
    taxalivrerisgoprom = 0.05
    sharpe = taxagananualprom/taxalivrerisgoprom
   
    return pd.Series({
        "Máximo": f"{maximo:.2f}",
        "Mínimo ": f"{minimo:.2f}",
        "Média": f"{media:.2f}",
        "Desvio padrão ": f"{desvio:.2f}",
        "Taxa de ganancia total ": f"{taxagantot * 100:.2f}%",
        "Taxa de ganancia anual prom ": f"{taxagananualprom * 100:.2f}%",
        "coef Sharpe": f"{sharpe:.2f}",
    })

In [171]:
resultado_index = estatisticas_index(df)
print(resultado_index.to_string())

Máximo                          137.25
Mínimo                           96.55
Média                           120.24
Desvio padrão                    11.63
Taxa de ganancia total          42.15%
Taxa de ganancia anual prom     12.56%
coef Sharpe                       2.51


In [77]:
def datas_drawdown_max(df):
    df = df.copy()
    df = df[df["index"].notna()]
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Série com índice acumulado
    acumulado = df["index"]
    pico = acumulado.cummax()
    drawdown = acumulado - pico

    # Índice do drawdown máximo
    idx_vale = drawdown.idxmin()
    idx_pico = (acumulado[:idx_vale]).idxmax()
    idx_trademin = df["trade"].idxmin()


    # Datas correspondentes
    data_pico = df.loc[idx_pico, "datetime"]
    data_vale = df.loc[idx_vale, "datetime"]
    data_trademin = df.loc[idx_trademin, "datetime"]

    # Diferença percentual
    valor_pico = df.loc[idx_pico, "index"]
    valor_vale = df.loc[idx_vale, "index"]
    drawdown_pct = ((valor_vale - valor_pico) / valor_pico) * 100
    trademin = df["trade"].min()
    
    return pd.Series({
        "Data do Pico": data_pico.strftime("%Y-%m-%d %H:%M"),
        "Data do Vale": data_vale.strftime("%Y-%m-%d %H:%M"),
        "Valor do Pico": round(valor_pico, 2),
        "Valor do Vale": round(valor_vale, 2),
        "Drawdown Máximo (%)": f"{drawdown_pct:.2f}%",
        "Màxima perdida por trade": f"{df["trade"].min()*100 :.2f}%",
        "Data do Max Loser Trade": data_trademin.strftime("%Y-%m-%d %H:%M"),
    })

In [79]:
resultado_datas = datas_drawdown_max(df)
print(resultado_datas)

Data do Pico                2019-07-01 14:00
Data do Vale                2022-05-12 09:00
Valor do Pico                          102.0
Valor do Vale                          57.41
Drawdown Máximo (%)                  -43.72%
Màxima perdida por trade              -5.23%
Data do Max Loser Trade     2021-10-07 08:00
dtype: object


In [173]:
def estatisticas_trades(df):
    df = df.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) * 100

    resultado = {
        "Total de Trades": len(trades),
        "Percentual de Trades Positivos (%)": f"{porcentagem_pos:.2f}%",
        "Média dos Trades Positivos": round(positivos.mean(), 6),
        "Desvio Padrão (Trades Positivos)": round(positivos.std(), 6),
        "Média dos Trades Negativos": round(negativos.mean(), 6),
        "Desvio Padrão (Trades Negativos)": round(negativos.std(), 6)
    }

    return pd.Series(resultado)

In [175]:
resumo_trades = estatisticas_trades(df)
print(resumo_trades)

Total de Trades                             60
Percentual de Trades Positivos (%)      16.67%
Média dos Trades Positivos            0.048725
Desvio Padrão (Trades Positivos)      0.044806
Média dos Trades Negativos           -0.016233
Desvio Padrão (Trades Negativos)      0.013479
dtype: object


In [85]:
def ranking_drawdowns_puros(df, coluna="index", top_n=5):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").head(top_n).reset_index(drop=True)

In [87]:
ranking_drawdowns_puros(df, coluna="index", top_n=5)


,Data Pico,Valor Pico,Data Vale,Valor Vale,Drawdown (%)
0,2019-07-01 14:00:00,102.004024,2022-05-12 09:00:00,57.409405,-43.72
1,2019-04-03 09:00:00,99.650000,2019-04-04 11:00:00,96.305464,-3.36
2,2019-04-15 14:00:00,100.271202,2019-06-26 13:00:00,97.024371,-3.24


In [89]:
def estatisticas_drawdowns_puros(df, coluna="index"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df[coluna].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            if vale_idx is not None and max_dd < 0:
                drawdowns.append(max_dd * 100)  # salva como porcentagem
            # novo ciclo
            pico = serie[i]
            pico_idx = i
            max_dd = 0
            vale_idx = None
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # salva último ciclo, se houver
    if vale_idx is not None and max_dd < 0:
        drawdowns.append(max_dd * 100)

    # série com drawdowns reais
    serie_dd = pd.Series(drawdowns)

    estatisticas = {
        "Total de Drawdowns": len(drawdowns),
        "Média dos Drawdowns (%)": round(serie_dd.mean(), 2),
        "Desvio Padrão (%)": round(serie_dd.std(), 2),
        "Drawdown Máximo (%)": round(serie_dd.min(), 2),
        "Drawdown Mínimo (%)": round(serie_dd.max(), 2)
    }

    return pd.Series(estatisticas)

In [91]:
resumo_dd_puros = estatisticas_drawdowns_puros(df)
print(resumo_dd_puros)

Total de Drawdowns          3.00
Média dos Drawdowns (%)   -16.77
Desvio Padrão (%)          23.34
Drawdown Máximo (%)       -43.72
Drawdown Mínimo (%)        -3.24
dtype: float64


In [177]:
def estatistica_periodos_estaticos_em_dias(df, coluna="index_sc"):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao_dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos_estaticos.append({
                "Valor index": grupo[coluna].iloc[0],
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "Duração (dias)": duracao_dias,
                "Número de Registros": len(grupo)
            })

    df_resultado = pd.DataFrame(periodos_estaticos)

    if df_resultado.empty:
        resumo = {
            "Total de Períodos Estáticos": 0,
            "Duração Média (dias)": 0,
            "Maior Duração (dias)": 0,
            "Data do Maior Período": {"Data Início": None, "Data Fim": None}
        }
    else:
        resumo = {
            "Total de Períodos Estáticos": len(df_resultado),
            "Duração Média (dias)": round(df_resultado["Duração (dias)"].mean(), 2),
            "Maior Duração (dias)": df_resultado["Duração (dias)"].max(),
            "Data do Maior Período": df_resultado.loc[df_resultado["Duração (dias)"].idxmax(), ["Data Início", "Data Fim"]].to_dict()
        }

    return df_resultado, pd.Series(resumo)

In [179]:
df_periodos, estatisticas = estatistica_periodos_estaticos_em_dias(df)
display(df_periodos)
display(estatisticas)

,Valor index,Data Início,Data Fim,Duração (dias),Número de Registros
0,101.833293,2016-04-25 11:00:00,2016-07-14 16:00:00,80,2
1,100.186849,2016-07-20 13:00:00,2016-08-09 14:00:00,20,2
2,99.646697,2016-08-11 11:00:00,2016-11-07 15:00:00,88,2
3,96.893201,2016-11-09 15:00:00,2017-02-07 12:00:00,89,2
4,101.054558,2017-02-16 15:00:00,2017-03-07 16:00:00,19,2
5,116.059841,2017-03-28 10:00:00,2017-04-06 13:00:00,9,2
6,114.084405,2017-04-10 13:00:00,2017-05-03 09:00:00,22,2
7,120.338571,2017-05-12 11:00:00,2017-05-31 13:00:00,19,2
8,122.288214,2017-06-07 11:00:00,2017-09-28 10:00:00,112,2
9,129.049297,2017-10-11 12:00:00,2017-10-17 14:00:00,6,2


Total de Períodos Estáticos                                                   29
Duração Média (dias)                                                       32.38
Maior Duração (dias)                                                         329
Data do Maior Período          {'Data Início': 2018-02-13 14:00:00, 'Data Fim...
dtype: object